In [ ]:
import pandas as pd
import huggingface_hub
from huggingface_hub import login
from datasets import load_dataset
from transformers import AutoTokenizer
import torch
from torch.utils.data import DataLoader
from transformers import AutoModelForMaskedLM, DataCollatorForLanguageModeling
from sklearn.model_selection import train_test_split
import os
import numpy as np

# os.environ["http_proxy"] = "http://127.0.0.1:10809"
# os.environ["https_proxy"] = "http://127.0.0.1:10809"



In [ ]:
# Upload dataset to huggingface
!huggingface-cli lfs-enable-largefiles

In [ ]:
login(token = "HF_TOKEN_REMOVED", add_to_git_credential=True)

### The repo is already created and do not need to be created again

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.create_repo(repo_id="soc-model")

### Upload the large file to the repo using Git LFS

In [ ]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="E:/Data/job_posting/processed/finetune/train_df.csv",
    path_in_repo="train_df.csv",
    repo_id="Zexuan/soc_data",
    repo_type="dataset",
)

In [10]:
from huggingface_hub import HfApi
api = HfApi()
api.upload_file(
    path_or_fileobj="C:/Users/lzx_l/cloud_share/Job_posting_data/Reviews.csv",
    path_in_repo="Reviews.csv",
    repo_id="Zexuan/soc_data",
    repo_type="dataset",
)

Reviews.csv:   0%|          | 0.00/301M [00:00<?, ?B/s]

Upload 1 LFS files:   0%|          | 0/1 [00:00<?, ?it/s]

'https://huggingface.co/datasets/Zexuan/soc_data/blob/main/Reviews.csv'

In [ ]:
remote_dataset = load_dataset("Zexuan/soc_data", split="train")

### Finetune a model

In [ ]:
train_df, test_df = train_test_split(pd.read_csv('F:/Data/job_posting/processed/finetune/df_titleLabel.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore'), test_size=0.3, random_state=42)

In [ ]:
# get the ONET SOC Code (the real one)
df_soc = pd.read_csv('F:/Data/job_posting/processed/2019_to_SOC_Crosswalk.csv')
# keep column '2018 SOC Code'
df_soc = df_soc[['2018 SOC Code']]
# replace the last digit of 'soc_code' with '0'
df_soc['2018 SOC Code'] = df_soc['2018 SOC Code'].str[:-1] + '0'
# rename the column name to 'soc_code'
df_soc.rename(columns={'2018 SOC Code': 'soc_code'}, inplace=True)
df_soc = df_soc.drop_duplicates(subset=['soc_code'], keep='first')
len(df_soc['soc_code'].unique())

In [ ]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification, AdamW, AutoModelForMaskedLM
from transformers import BertForSequenceClassification
import torch
from transformers import AdamW
from transformers import get_scheduler
from transformers import Trainer
from transformers import BertTokenizer
import numpy as np
import pandas as pd


# model = AutoModelForSequenceClassification.from_pretrained("bert-base-chinese")
# model = BertForSequenceClassification.from_pretrained("hfl/chinese-bert-wwm")

#tokenizer = AutoTokenizer.from_pretrained("bert-base-chinese")
tokenizer = BertTokenizer.from_pretrained("bert-base-chinese")

In [ ]:
train_df_sample = pd.read_csv('F:/Data/job_posting/processed/finetune/train_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
# replace the last digit of 'soc_code' with '0'
train_df_sample['soc_code'] = train_df_sample['soc_code'].str[:-1] + '0'
# merge the 'test_dfSoc' and 'df_soc' using 'soc_code', only keep the matched sample
train_df_sample = pd.merge(train_df_sample, df_soc, on='soc_code', how='inner')

In [ ]:
test_df_sample = pd.read_csv('F:/Data/job_posting/processed/finetune/test_df.csv', encoding = "utf_8_sig", on_bad_lines='skip', encoding_errors='ignore')
# replace the last digit of 'soc_code' with '0'
test_df_sample['soc_code'] = test_df_sample['soc_code'].str[:-1] + '0'
# merge the 'test_dfSoc' and 'df_soc' using 'soc_code', only keep the matched sample
test_df_sample = pd.merge(test_df_sample, df_soc, on='soc_code', how='inner')

In [ ]:
# keep the overalp soc_code in train and test dataset
train_df_sample = train_df_sample[train_df_sample['soc_code'].isin(test_df_sample['soc_code'])]
test_df_sample = test_df_sample[test_df_sample['soc_code'].isin(train_df_sample['soc_code'])]

# replace the symbol '-' to '.' in soc_code column, and convert soc_code to int
train_df_sample['soc_code'] = train_df_sample['soc_code'].str.replace('-', '').astype(int)
test_df_sample['soc_code'] = test_df_sample['soc_code'].str.replace('-', '').astype(int)

# generate a new column 'soc_code1' with value to recode the 'soc_code' in ascending order
train_df_sample['soc_code1'] = train_df_sample['soc_code'].rank(method='dense').astype(int)
test_df_sample['soc_code1'] = test_df_sample['soc_code'].rank(method='dense').astype(int)

# recode the 'soc_code 1' so that it starts from value 0
train_df_sample['soc_code1'] = train_df_sample['soc_code1'] - train_df_sample['soc_code1'].min()
test_df_sample['soc_code1'] = test_df_sample['soc_code1'] - test_df_sample['soc_code1'].min()

# convert '工作描述' to string
train_df_sample['工作描述'] = train_df_sample['工作描述'].astype(str)
test_df_sample['工作描述'] = test_df_sample['工作描述'].astype(str)


In [ ]:

# Tokenize the text and convert it into input features
train_texts = train_df_sample['工作描述'].tolist()
train_labels = train_df_sample['soc_code1'].tolist()
train_encodings = tokenizer(train_texts, truncation=True, padding=True)

test_texts = test_df_sample['工作描述'].tolist()
test_labels = test_df_sample['soc_code1'].tolist()
test_encodings = tokenizer(test_texts, truncation=True, padding=True)

In [ ]:
import torch
from torch.utils.data import DataLoader, TensorDataset
from transformers import BertForSequenceClassification, AdamW, get_linear_schedule_with_warmup

# Convert the input features into PyTorch tensors
train_inputs = torch.tensor(train_encodings['input_ids'])
train_masks = torch.tensor(train_encodings['attention_mask'])
train_labels = torch.tensor(train_labels)

test_inputs = torch.tensor(test_encodings['input_ids'])
test_masks = torch.tensor(test_encodings['attention_mask'])
test_labels = torch.tensor(test_labels)


In [ ]:
# Create a PyTorch DataLoader to iterate over the data during training
batch_size = 64

train_data = TensorDataset(train_inputs, train_masks, train_labels)
train_loader = DataLoader(train_data, batch_size=batch_size, shuffle=True)

test_data = TensorDataset(test_inputs, test_masks, test_labels)
test_loader = DataLoader(test_data, batch_size=batch_size)

# Define the model, the optimizer and the learning rate scheduler
num_labels = len(train_df_sample['soc_code1'].unique())
model = BertForSequenceClassification.from_pretrained("hfl/chinese-bert-wwm", num_labels=num_labels)
optimizer = AdamW(model.parameters(), lr=2e-5, eps=1e-8)
num_epochs = 4
num_training_steps = num_epochs * len(train_loader)
scheduler = get_linear_schedule_with_warmup(optimizer, num_warmup_steps=0, num_training_steps=num_training_steps)

# Train the model
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device)
model.train()
for epoch in range(num_epochs):
    for batch in train_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)
        labels = batch[2].to(device)

        outputs = model(inputs, attention_mask=masks, labels=labels)
        loss = outputs.loss
        loss.backward()

        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)

        optimizer.step()
        scheduler.step()
        optimizer.zero_grad()


In [ ]:
# Evaluate the performance of the model on the test set
model.eval()
predictions = []
with torch.no_grad():
    for batch in test_loader:
        inputs = batch[0].to(device)
        masks = batch[1].to(device)

        # Resize the attention mask tensor to match the size of the inputs
        # masks.resize_(inputs.shape[0], inputs.shape[1])
    
        outputs = model(inputs, attention_mask=masks)
        logits = outputs.logits
        batch_predictions = torch.argmax(logits, axis=1).cpu().numpy()
        predictions.extend(batch_predictions)

from sklearn.metrics import accuracy_score
accuracy = accuracy_score(test_labels, predictions)
print(f"Accuracy: {accuracy}")

In [ ]:
print(train_inputs.shape)

In [ ]:
# labels = labels.unsqueeze(1)
print(train_labels.shape)